In [10]:
import pandas as pd
import numpy as np

In [11]:
def run_boosting_ensemble_ppt_style(model_files, k=8):
    """
    依照 PPT 範例實作：迭代更新題目權重 w_i 與模型權重 alpha_j。
    包含：
    1. 題目權重與模型權重每輪正規化為 1.0
    2. Alpha 最小值限制為 0.0001
    """
    models_data = {}
    for name, file in model_files.items():
        models_data[name] = pd.read_excel(file)
    
    num_instances = len(next(iter(models_data.values())))
    actual_labels = next(iter(models_data.values()))['Actual']
    
    # 1. 初始設定：題目初始權重每題平等並正規化 [cite: 619]
    instance_weights = np.ones(num_instances) / num_instances
    
    # 儲存各模型最終權重 [cite: 663-666]
    final_cumulative_alphas = {name: 0.0 for name in model_files.keys()}

    print(f"--- 執行 Boosting 迭代 (設定輪數 k={k}, Alpha 門檻=0.0001) ---")

    for round_idx in range(1, k + 1):
        print(f"\n【第 {round_idx} 輪迭代】")
        
        current_round_alphas = {}
        
        for name, df in models_data.items():
            # 取得該模型預測錯誤的遮罩 (1為錯, 0為對) [cite: 621-622]
            errors = (df['Correct'] == False).astype(int)
            
            # 2. 計算加權誤差 Ej [cite: 625-626, 635-636]
            weighted_error = np.dot(instance_weights, errors) / np.sum(instance_weights)
            # 避免除以 0 的極端數值處理
            weighted_error = max(min(weighted_error, 0.99999), 0.00001)
            
            # 3. 計算本輪投票權重 alpha_j [cite: 627-628, 637-638]
            alpha = 0.5 * np.log((1 - weighted_error) / weighted_error)
            
            # --- 套用用戶約束：Alpha 不得小於 0.0001 ---
            if alpha < 0.0001:
                alpha = 0.0001
                
            current_round_alphas[name] = alpha
            
            # 4. 更新題目權重: 做錯的題目權重增加 alpha [cite: 629-630, 639-640, 673]
            instance_weights = instance_weights + alpha *  0.001 * errors
            
            # --- 題目權重正規化 (使總和為 1.0) ---
            instance_weights = instance_weights / np.sum(instance_weights)
            
            print(f"  > 評估 {name}: Ej={weighted_error:.4f}, 修正後 Alpha={alpha:.4f}")

        # --- 模型權重正規化 (使本輪 Alpha 總和為 1.0) --- [cite: 261-263]
        total_alpha = sum(current_round_alphas.values())
        if total_alpha > 0:
            for name in current_round_alphas:
                final_cumulative_alphas[name] = current_round_alphas[name] / total_alpha

        # 每輪結束後計算當前正確率 
        current_correct = 0
        for idx in range(num_instances):
            votes = {}
            for name in final_cumulative_alphas:
                pred = models_data[name].loc[idx, 'Predicted']
                votes[pred] = votes.get(pred, 0) + final_cumulative_alphas[name]
            
            # 取得得分最高者作為集成預測 [cite: 257-258, 294]
            final_pred = max(votes, key=votes.get)
            if final_pred == actual_labels[idx]:
                current_correct += 1
        
        acc = (current_correct / num_instances) * 100
        print(f"  > 本輪正規化權重: " + ", ".join([f"{n}: {w:.4f}" for n, w in final_cumulative_alphas.items()]))
        print(f"  > 目前集成正確率: {current_correct}/{num_instances} = {acc:.2f}%")

    return final_cumulative_alphas, models_data

In [12]:
# --- 主程式執行 ---
files = {
    "M1": "result_huihui_ai_internlm3-abliterated.xlsx",
    "M2": "result_qwen3_8b.xlsx",
    "M3": "result_glm4_9b.xlsx"
}

In [13]:
# 依照範例設定 k=20
final_weights, data = run_boosting_ensemble_ppt_style(files, k=20)

--- 執行 Boosting 迭代 (設定輪數 k=20, Alpha 門檻=0.0001) ---

【第 1 輪迭代】
  > 評估 M1: Ej=0.0589, 修正後 Alpha=1.3852
  > 評估 M2: Ej=0.1357, 修正後 Alpha=0.9258
  > 評估 M3: Ej=0.1848, 修正後 Alpha=0.7420
  > 本輪正規化權重: M1: 0.4537, M2: 0.3032, M3: 0.2430
  > 目前集成正確率: 1893/2002 = 94.56%

【第 2 輪迭代】
  > 評估 M1: Ej=0.2507, 修正後 Alpha=0.5475
  > 評估 M2: Ej=0.2795, 修正後 Alpha=0.4735
  > 評估 M3: Ej=0.2939, 修正後 Alpha=0.4383
  > 本輪正規化權重: M1: 0.3752, M2: 0.3245, M3: 0.3004
  > 目前集成正確率: 1893/2002 = 94.56%

【第 3 輪迭代】
  > 評估 M1: Ej=0.3195, 修正後 Alpha=0.3781
  > 評估 M2: Ej=0.3457, 修正後 Alpha=0.3190
  > 評估 M3: Ej=0.3520, 修正後 Alpha=0.3052
  > 本輪正規化權重: M1: 0.3772, M2: 0.3182, M3: 0.3045
  > 目前集成正確率: 1893/2002 = 94.56%

【第 4 輪迭代】
  > 評估 M1: Ej=0.3609, 修正後 Alpha=0.2858
  > 評估 M2: Ej=0.3862, 修正後 Alpha=0.2317
  > 評估 M3: Ej=0.3890, 修正後 Alpha=0.2258
  > 本輪正規化權重: M1: 0.3845, M2: 0.3118, M3: 0.3037
  > 目前集成正確率: 1893/2002 = 94.56%

【第 5 輪迭代】
  > 評估 M1: Ej=0.3890, 修正後 Alpha=0.2258
  > 評估 M2: Ej=0.4135, 修正後 Alpha=0.1748
  > 評估 M3: Ej=0.4146, 修正後 A